# 质量因数## 谐振电路和 Q 值谐振电路广泛应用于振荡器、调谐放大器、滤波器等。在特定频率下，即_谐振频率_ $f_r$（或 $\omega_r$），谐振器呈现出最大（或最小）阻抗（例如：开路或短路）[1]。质量因数 $Q$，或简称 Q 值，是描述谐振无源电路损耗的无量纲指标，定义如下[2, 3]：$$Q = 2 \pi \left. \frac{\textrm{平均存储能量}}{\textrm{每秒能量损耗}} \right|_{\omega=\omega_r}=\omega_r \left. \frac{\textrm{平均存储能量}}{\textrm{平均功率损耗}} \right|_{\omega=\omega_r}$$从这个定义可以看出，较低的损耗意味着较高的 $Q$ 值。具有较高 Q 值的谐振器在谐振频率处具有更大的振幅，但其谐振频率附近的频率范围 $BW$ 较小。## 负载 Q 值和无负载 Q 值实际上，可以根据考虑的损耗类型，定义三种 Q 值[2]：* 无负载 Q 值：$$Q_0 = \omega_r \left. \frac{\textrm{存储在谐振电路中的能量}}{\textrm{谐振电路中的功率损耗}} \right|_{\omega=\omega_r}$$* 外部 Q 值：$$Q_e = \omega_r \left. \frac{\textrm{存储在谐振电路中的能量}}{\textrm{外部电路中的功率损耗}} \right|_{\omega=\omega_r}$$* 负载 Q 值：$$Q_L = \omega_r \left. \frac{\textrm{存储在谐振电路中的能量}}{\textrm{总功率损耗}} \right|_{\omega=\omega_r}$$_负载_ Q 值 $Q_L$，描述了整个谐振系统中的能量耗散，该系统包括谐振器本身和用于观察谐振的仪器[3]。术语_加载_指的是外部电路对测量量的影响。由于外部电路引起的任何损耗机制都会降低 $Q$ 值。_无负载_ Q 值 $Q_0$ 是谐振器本身的特性，不受外部电路引起的任何加载效应的影响（不耦合）。对于大多数应用，所需的量是无负载 Q 值 $Q_0$，它仅由与谐振器相关的能量耗散决定，因此可以最好地描述谐振模式。通常无法直接测量谐振器的无负载 Q 值，因为测量系统会产生加载效应。但是，可以通过测量加载谐振器的频率响应来估计 $Q_0$。外部电路中的能量耗散由_外部_ Q 值 $Q_e$ 描述，这三个 Q 值通过以下关系相关（从上述三个表达式推导得出）：$$\frac{1}{Q_L} = \frac{1}{Q_e} + \frac{1}{Q_0} $$如果定义_耦合系数_ $\beta = Q_0/Q_e$，则：$$Q_0 = (1 + \beta) Q_L$$可以区分三种情况：1. $\beta<1$：谐振器被认为是_欠耦合_到馈电电路。2. $\beta=1$：谐振器被认为是_临界耦合_。3. $\beta>1$：谐振器被认为是_过耦合_。虽然测量负载质量因数 $Q_L$ 是一个简单直接的过程，但要获得低于 1% 的不确定度（这被认为是 Q 值测量的低不确定度），需要注意实验过程的几个方面。此外，从测量的 S 参数中找到无负载 $Q_0$，首先需要找到耦合系数 $\beta$，然后从 3dB 带宽中测量 $Q_L$，并使用上述关系。幸运的是，`scikit-rf` 实现了从频域 S 参数自动确定_负载_和_无负载_ Q 值的方法。所实现的方法在 [4] 中详细描述，并且可以应用于传输或反射的测量。## 并联 RLC 电路的示例为了说明 `scikit-rf` 中 `Qfactor` 类的用法，使用上面图示的并联 RLC 电路作为示例，该电路具有可用于基准测试的分析公式。<img src="figures/Parallel_RLC_resonator.svg" width="300"/>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf

rf.stylely()

C = 1e-6  # FL = 1e-9  # HR = 30  # OhmZ0 = 50  # Ohmfreq = rf.Frequency(5, 5.2, npoints=501, unit='MHz')media = rf.media.DefinedGammaZ0(frequency=freq, z0=Z0)  # ideal line (no loss)rng = np.random.default_rng()random_d = rng.uniform(-np.pi, np.pi)  # a random length for the sake of the exampleresonator = media.line(d=random_d, unit='rad') \                **media.shunt_inductor(L) ** media.shunt_capacitor(C) \                ** media.shunt(media.resistor(R)**media.short()) ** media.open()resonator.plot_s_db()

In [ ]:
C = 1e-6  # F
L = 1e-9  # H
R = 30  # Ohm
Z0 = 50  # Ohm

freq = rf.Frequency(5, 5.2, npoints=501, unit='MHz')
media = rf.media.DefinedGammaZ0(frequency=freq, z0=Z0)  # ideal line (no loss)
rng = np.random.default_rng()
random_d = rng.uniform(-np.pi, np.pi)  # a random length for the sake of the example

resonator = media.line(d=random_d, unit='rad') \
                **media.shunt_inductor(L) ** media.shunt_capacitor(C) \
                ** media.shunt(media.resistor(R)**media.short()) ** media.open()

resonator.plot_s_db()

def f_res_RLC(L, C):    return 1/(2*np.pi*np.sqrt(L*C))def Q_RLC(R, L, C):    return R * C / np.sqrt(L*C)

In [ ]:
def f_res_RLC(L, C):
    return 1/(2*np.pi*np.sqrt(L*C))

def Q_RLC(R, L, C):
    return R * C / np.sqrt(L*C)

print(f'Theoretical Resonant Frequency: {f_res_RLC(L, C)/1e6} MHz')print(f'Theoretical Loaded Q: Q_L = {Q_RLC((R*Z0)/(R+Z0), L, C)}')  # Req = R//Z0print(f'Theoretical Unloaded Q: Q_0 = {Q_RLC(R, L, C)}')

In [ ]:
print(f'Theoretical Resonant Frequency: {f_res_RLC(L, C)/1e6} MHz')
print(f'Theoretical Loaded Q: Q_L = {Q_RLC((R*Z0)/(R+Z0), L, C)}')  # Req = R//Z0
print(f'Theoretical Unloaded Q: Q_0 = {Q_RLC(R, L, C)}')

Q = rf.qfactor.Qfactor(resonator, res_type='reflection')

In [ ]:
Q = rf.qfactor.Qfactor(resonator, res_type='reflection')

res = Q.fit()print(f'Fitted Resonant Frequency: f_L = {Q.f_L/1e6} MHz')print(f'Fitted Loaded Q-factor: Q_L = {Q.Q_L}')

In [ ]:
res = Q.fit()
print(f'Fitted Resonant Frequency: f_L = {Q.f_L/1e6} MHz')
print(f'Fitted Loaded Q-factor: Q_L = {Q.Q_L}')

Q

In [ ]:
Q

Q0 = Q.Q_unloaded(res)print(f'Fitted Unloaded Q-factor: Q_0 = {Q0}')

In [ ]:
Q0 = Q.Q_unloaded(res)
print(f'Fitted Unloaded Q-factor: Q_0 = {Q0}')

Q0 = Q.Q_unloaded()  # will use the latest optimized results performed with .fit()print(f'Fitted Unloaded Q-factor: Q_0 = {Q0}')

In [ ]:
Q0 = Q.Q_unloaded()  # will use the latest optimized results performed with .fit()
print(f'Fitted Unloaded Q-factor: Q_0 = {Q0}')

print('Relative Error on Q_0:', (Q_RLC(R, L, C) - Q0)/Q_RLC(R, L, C))

In [ ]:
print('Relative Error on Q_0:', (Q_RLC(R, L, C) - Q0)/Q_RLC(R, L, C))

new_freq = rf.Frequency(5, 5.2, npoints=5001, unit='MHz')fitted_network = Q.fitted_network(res, frequency=new_freq)

In [ ]:
new_freq = rf.Frequency(5, 5.2, npoints=5001, unit='MHz')
fitted_network = Q.fitted_network(res, frequency=new_freq)

In [ ]:
resonator.plot_s_mag(label='Parallel RLC ', lw=2)
fitted_network.plot_s_mag(label='Fitted Model', lw=2, ls='--')

Q 圆的参数（直径以及调谐和失谐的散射参数）最终可以从以下途径获得：

diam, S_V, S_T = Q.Q_circle()

In [ ]:
diam, S_V, S_T = Q.Q_circle()

fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})resonator.plot_s_polar(ax=ax, label="RLC Resonator", ls='', marker='x', ms=5)fitted_network.plot_s_polar(ax=ax, label="Fitted Model", lw=2)ax.plot(np.angle(S_V), np.abs(S_V), 'ko')ax.plot(np.angle(S_T), np.abs(S_T), 'ko')ax.text(np.angle(S_T), 0.8*np.abs(S_T), '$S_T$')ax.text(np.angle(S_V), 1.1*np.abs(S_V), '$S_V$')

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
resonator.plot_s_polar(ax=ax, label="RLC Resonator", ls='', marker='x', ms=5)
fitted_network.plot_s_polar(ax=ax, label="Fitted Model", lw=2)
ax.plot(np.angle(S_V), np.abs(S_V), 'ko')
ax.plot(np.angle(S_T), np.abs(S_T), 'ko')
ax.text(np.angle(S_T), 0.8*np.abs(S_T), '$S_T$')
ax.text(np.angle(S_V), 1.1*np.abs(S_V), '$S_V$')

BW = Q.BWprint(f'Bandwidth: {BW} Hz')

In [ ]:
BW = Q.BW
print(f'Bandwidth: {BW} Hz')

In [ ]:
fig, ax = plt.subplots()
resonator.plot_s_db(label='Parallel RLC ', lw=2, ax=ax)
ax.axvspan(xmin=Q.f_L-Q.BW/2, xmax=Q.f_L+Q.BW/2, alpha=0.3, label='Bandwidth')
ax.legend()

## References
- [1] B. A. Galwas, "Scattering Matrix Description of Microwave Resonators," in IEEE Transactions on Microwave Theory and Techniques, vol. 31, no. 8, pp. 669-671, Aug. 1983, doi: 10.1109/TMTT.1983.1131566.
- [2] Peter A. Rizzi, "Microwave Engineering: Passive Circuits", Prentice-Hall, 1988 
- [3] David M. Pozar, "Microwave Engineering", 4th Edition, Éditeur	Wiley, 2011
- [4] A. P. Gregory, "Q-factor Measurement by using a Vector Network Analyser", National Physical Laboratory Report MAT 58 (2021), https://eprintspublications.npl.co.uk/9304/
- [5] Microwaves101, "Resonance of RLC Circuits", https://www.microwaves101.com/encyclopedias/resonance-of-rlc-circuits
- [6] Green, Estill I. (October 1955). "The Story of Q", American Scientist. 43: 584–594. http://www.collinsaudio.com/Prosound_Workshop/The_story_of_Q.pdf
- [7] R. S. Kwok and J.-F. Liang, ‘Characterization of high-Q resonators for microwave filter applications’, IEEE Transactions on Microwave Theory and Techniques, vol. 47, no. 1, pp. 111–114, Jan. 1999, doi: 10.1109/22.740093.
- [8] Darko Kajfez, "Q factor measurements, analog and digital", https://people.engineering.olemiss.edu/darko-kajfez/assets/rfqmeas2b.pdf
